# mini-vLLM: a serving loop you can take apart

**CPU scheduler · simulated compute · no model weights or GPU.**

This notebook runs the actual mini-vLLM Python scheduler and block manager. The browser terminal is a smaller JavaScript teaching model; this notebook is the bridge to the repository implementation. Output-token counts and compute time are synthetic. They are not vLLM/GPU benchmarks.

Run cells top to bottom in a Python 3.10+ Jupyter kernel. The first cell installs a pinned public source revision if mini-vLLM is not already available. No LLM service is started.

In [1]:
import importlib.util, subprocess, sys
if importlib.util.find_spec("mini_vllm") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "git+https://github.com/WeishuZ/mini-vllm.git@4589d049bac4fd7bb5a21f99944a49742e8b64b3"])
print("mini-vLLM CPU scheduler ready; compute is simulated.")
print("Reference source revision: 4589d049bac4fd7bb5a21f99944a49742e8b64b3")

mini-vLLM CPU scheduler ready; compute is simulated.
Reference source revision: 4589d049bac4fd7bb5a21f99944a49742e8b64b3


## 1. Step through a request

A physical block holds `block_size` token slots. Prefill materializes prompt KV. A decode step then emits one token per ready sequence and grows its block table as needed. `engine.run(max_steps=...)` lets us advance the public engine API one iteration at a time.

In [2]:
from mini_vllm import LLMEngine, CacheConfig, SchedulerConfig, ModelConfig, Request

def make_engine(policy="continuous", blocks=12):
    return LLMEngine(
        CacheConfig(block_size=16, num_gpu_blocks=blocks, enable_prefix_caching=False),
        SchedulerConfig(policy=policy, max_num_seqs=4, max_num_batched_tokens=32, watermark=0.04),
        ModelConfig(),
    )

engine = make_engine()
engine.add_request(Request("req-01", prompt_len=48, max_tokens=16))
engine.add_request(Request("req-02", prompt_len=32, max_tokens=10))
for _ in range(6):
    engine.run(max_steps=engine.num_steps + 1)
    print(f"step={engine.num_steps:02} simulated_ms={engine.clock_ms:.1f}")
    for seq in engine.sequences.values():
        print(seq.request_id, seq.status.value, "computed=", seq.num_computed,
              "generated=", seq.num_generated, "block_table=", seq.block_table)
print("All timing above comes from ModelRunner's deterministic cost function.")

step=01 simulated_ms=3.7
req-01 running computed= 32 generated= 0 block_table= [11, 10]
req-02 waiting computed= 0 generated= 0 block_table= []
step=02 simulated_ms=7.4
req-01 running computed= 48 generated= 0 block_table= [11, 10, 9]
req-02 running computed= 16 generated= 0 block_table= [8]
step=03 simulated_ms=15.8
req-01 running computed= 49 generated= 1 block_table= [11, 10, 9, 7]
req-02 running computed= 32 generated= 0 block_table= [8, 6]
step=04 simulated_ms=23.1
req-01 running computed= 50 generated= 2 block_table= [11, 10, 9, 7]
req-02 running computed= 33 generated= 1 block_table= [8, 6, 5]
step=05 simulated_ms=30.2
req-01 running computed= 51 generated= 3 block_table= [11, 10, 9, 7]
req-02 running computed= 34 generated= 2 block_table= [8, 6, 5]
step=06 simulated_ms=37.5
req-01 running computed= 52 generated= 4 block_table= [11, 10, 9, 7]
req-02 running computed= 35 generated= 3 block_table= [8, 6, 5]
All timing above comes from ModelRunner's deterministic cost function.


## 2. Finish and check resource ownership

Finished requests return their blocks. The compute model has no transformer forward pass; changing it changes simulated timings, not GPU measurements.

In [3]:
metrics = engine.run()
assert metrics.num_completed == 2
assert len(engine.scheduler.running) == 0
assert all(not seq.block_table for seq in engine.completed)
print(metrics.summary())
print("Completed block tables:", [seq.block_table for seq in engine.completed])

requests=2/2  throughput=  212.9 tok/s  queue p50=   1.9 ms  TTFT p50/p99=  19.4/  23.0 ms  ITL p50=  7.2 ms  e2e p50=  105.0 ms  preempt=0  swap=0  peakKV=58.3%  cache_hit= 0.0%  saved_prefill=0
Completed block tables: [[], []]


## 3. Continuous versus static batching

Use identical requests and the same synthetic compute cost. Compare how quickly free batch slots can admit waiting work. This is a model experiment, not an expected speedup on real hardware.

In [4]:
requests = [Request(f"batch-{i}", prompt_len=p, max_tokens=g)
            for i, (p, g) in enumerate([(48, 24), (16, 8), (64, 16), (32, 12)])]
for policy in ["static", "continuous"]:
    candidate = make_engine(policy)
    candidate.add_requests(requests)
    result = candidate.run(max_steps=2000)
    assert result.num_completed == len(requests)
    print(policy, result.summary())

static requests=4/4  throughput=  220.2 tok/s  queue p50=   0.0 ms  TTFT p50/p99=  26.8/ 192.3 ms  ITL p50=  7.2 ms  e2e p50=  166.0 ms  preempt=0  swap=0  peakKV=100.0%  cache_hit= 0.0%  saved_prefill=0
continuous requests=4/4  throughput=  303.2 tok/s  queue p50=   5.6 ms  TTFT p50/p99=  26.8/  89.8 ms  ITL p50=  7.5 ms  e2e p50=  176.1 ms  preempt=1  swap=0  peakKV=100.0%  cache_hit= 0.0%  saved_prefill=0


## 4. Memory pressure and recomputation

A deliberately small pool can force the scheduler to discard lower-priority KV and recompute it later. The sequence keeps generated tokens, which become prefill work after eviction. Try changing the block budget and inspect preemption counts.

In [5]:
pressure = make_engine(blocks=8)
pressure.add_requests([Request(f"pressure-{i}", prompt_len=48, max_tokens=48) for i in range(3)])
result = pressure.run(max_steps=2000)
assert result.num_completed == 3
assert pressure.num_preemptions > 0
print(result.summary())
for seq in pressure.completed:
    print(seq.request_id, "recomputations=", seq.num_preemptions)
print("These recomputations are from the CPU model, not a production incident.")

requests=3/3  throughput=  142.5 tok/s  queue p50=   3.7 ms  TTFT p50/p99=  24.7/ 675.6 ms  ITL p50=  6.9 ms  e2e p50=  679.9 ms  preempt=29  swap=0  peakKV=100.0%  cache_hit= 0.0%  saved_prefill=0
pressure-0 recomputations= 0
pressure-1 recomputations= 16
pressure-2 recomputations= 13
These recomputations are from the CPU model, not a production incident.


## 5. Map the experiment to vLLM

| Here | Real vLLM reference |
|---|---|
| Running / waiting queues | `vllm:num_requests_running`, `vllm:num_requests_waiting` |
| Used KV blocks / total blocks | `vllm:kv_cache_usage_perc` (ratio; 1 means full) |
| Per-step token budget | `max_num_batched_tokens` / `--max-num-batched-tokens` |
| Recompute count | `vllm:num_preemptions` |
| First generated token timestamp | `vllm:time_to_first_token_seconds` |

The browser/CPU lab omits actual GPU attention kernels and uses a deterministic latency function. Its small-scale settings and timings are not real-vLLM tuning recommendations. vLLM V1 prioritizes decode work before filling the remaining token budget with chunked prefill, and normally uses recomputation for preemption.

Read the primary sources: [vLLM metrics](https://docs.vllm.ai/en/latest/usage/metrics/), [V1 scheduling and tuning](https://docs.vllm.ai/en/latest/configuration/optimization/), [mini-vLLM source](https://github.com/WeishuZ/mini-vllm).

Suggested experiment: halve the token budget, rerun the same requests, and explain which waiting time, prefill progress and simulated TTFT change. Then increase the KV pool and explain which recomputations disappear.